STAGE 4.5
# OFFICIAL PROTOTYPE → TEST RETRIEVAL EVALUATION

In [2]:
# =============================================================================
# PARASURG / CISLR RESEARCH
# STAGE 4.5 — OFFICIAL CISLR EVALUATION
# =============================================================================
#
# IMPORTANT:
#   - NO MODEL TRAINING
#   - NO CHECKPOINT MODIFICATION
#   - NO RAW VIDEO MODIFICATION
#   - Uses official prototype -> test retrieval
#
# Robust video decoding:
#   - sequential decode from frame 0
#   - never seeks to requested frame positions
#   - handles truncated/corrupt videos when some frames remain decodable
#   - uniformly samples 8 frames from ACTUALLY decoded frames
#
# Evaluation representation:
#   - penultimate backbone embedding
#   - mean over 8 frame embeddings
#   - L2 normalization
#   - cosine similarity against all 4764 prototypes
#
# Models:
#   - ResNet18 Stage 4.4 v2
#   - MobileNetV3-Large Stage 4.4 v2
#   - EfficientNet-B0 Stage 4.4 exact mean-logit v2
#
# =============================================================================


# =============================================================================
# 0. IMPORTS
# =============================================================================

import os
import json
import time
import math
import hashlib
import warnings
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms

from tqdm.auto import tqdm


# =============================================================================
# 1. GLOBAL CONFIGURATION
# =============================================================================

PROJECT_ROOT = Path("/home/dipshikha/Music/cao")

VIDEO_ROOT = PROJECT_ROOT / "CISLR_v1.5-a_videos"

DATASET_CSV = PROJECT_ROOT / "dataset.csv"
PROTOTYPE_CSV = PROJECT_ROOT / "prototype.csv"
TEST_CSV = PROJECT_ROOT / "test.csv"

RESEARCH_ROOT = PROJECT_ROOT / "CISLR_RESEARCH"

MODEL_ROOT = RESEARCH_ROOT / "models" / "stage4_4"

STAGE45_ROOT = RESEARCH_ROOT / "audit" / "stage4_5_official_evaluation"

EMBEDDING_ROOT = STAGE45_ROOT / "embeddings"
PREDICTION_ROOT = STAGE45_ROOT / "predictions"
SIMILARITY_ROOT = STAGE45_ROOT / "similarity"
METRIC_ROOT = STAGE45_ROOT / "metrics"

for directory in [
    STAGE45_ROOT,
    EMBEDDING_ROOT,
    PREDICTION_ROOT,
    SIMILARITY_ROOT,
    METRIC_ROOT,
]:
    directory.mkdir(parents=True, exist_ok=True)


# -----------------------------------------------------------------------------
# Evaluation configuration
# -----------------------------------------------------------------------------

NUM_FRAMES = 8
IMAGE_SIZE = 224

NUM_CLASSES = 4764

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


print("=" * 80)
print("STAGE 4.5 — OFFICIAL CISLR EVALUATION")
print("=" * 80)
print(f"Project root : {PROJECT_ROOT}")
print(f"Video root   : {VIDEO_ROOT}")
print(f"Device       : {DEVICE}")
print(f"Num classes  : {NUM_CLASSES}")
print(f"Frames/video : {NUM_FRAMES}")
print(f"Image size   : {IMAGE_SIZE}")
print("=" * 80)


# =============================================================================
# 2. CHECK REQUIRED FILES
# =============================================================================

required_files = [
    DATASET_CSV,
    PROTOTYPE_CSV,
    TEST_CSV,

    MODEL_ROOT / "resnet18_stage4_4_v2_best.pt",
    MODEL_ROOT / "mobilenet_v3_large_v2" /
        "mobilenet_v3_large_stage4_4_v2_best.pt",
    MODEL_ROOT / "efficientnet_b0_v2_exact_meanlogit" /
        "efficientnet_b0_stage4_4_v2_exact_meanlogit_best.pt",
]

print("\nChecking required files...")

for path in required_files:
    if not path.exists():
        raise FileNotFoundError(f"Required file not found:\n{path}")
    print(f"[OK] {path}")

print("\nAll required files found.")


# =============================================================================
# 3. LOAD DATASET METADATA
# =============================================================================

print("\nLoading dataset metadata...")

dataset_df = pd.read_csv(DATASET_CSV)
prototype_df = pd.read_csv(PROTOTYPE_CSV)
test_df = pd.read_csv(TEST_CSV)

print(f"dataset.csv   : {dataset_df.shape}")
print(f"prototype.csv : {prototype_df.shape}")
print(f"test.csv      : {test_df.shape}")


# =============================================================================
# 4. CLEAN INVALID LABEL ROW
# =============================================================================

INVALID_UID = "R_GaklgD4bY=-"

dataset_df = dataset_df[
    dataset_df["uid"].astype(str) != INVALID_UID
].copy()

prototype_df = prototype_df[
    prototype_df["uid"].astype(str) != INVALID_UID
].copy()

test_df = test_df[
    test_df["uid"].astype(str) != INVALID_UID
].copy()

print("\nAfter removing invalid/malformed UID:")
print(f"dataset   : {len(dataset_df)}")
print(f"prototype : {len(prototype_df)}")
print(f"test      : {len(test_df)}")


# =============================================================================
# 5. BASIC OFFICIAL SPLIT VALIDATION
# =============================================================================

prototype_uids = set(prototype_df["uid"].astype(str))
test_uids = set(test_df["uid"].astype(str))

overlap = prototype_uids.intersection(test_uids)

if len(overlap) != 0:
    raise RuntimeError(
        f"Prototype/test leakage detected: {len(overlap)} overlapping UIDs."
    )

print("\n[OK] Prototype/Test UID overlap = 0")


# =============================================================================
# 6. BUILD GLOSS -> CLASS-ID MAPPING
# =============================================================================

all_valid_glosses = sorted(
    dataset_df["gloss"]
    .dropna()
    .astype(str)
    .unique()
)

if len(all_valid_glosses) != NUM_CLASSES:
    raise RuntimeError(
        f"Expected {NUM_CLASSES} unique gloss classes, "
        f"found {len(all_valid_glosses)}."
    )

gloss_to_class = {
    gloss: idx
    for idx, gloss in enumerate(all_valid_glosses)
}

class_to_gloss = {
    idx: gloss
    for gloss, idx in gloss_to_class.items()
}


# =============================================================================
# 7. RESOLVE VIDEO PATH
# =============================================================================

def resolve_video_path(uid):
    """
    Resolve UID to an MP4 path.

    Searches the known CISLR video root.
    """

    uid = str(uid)

    direct = VIDEO_ROOT / f"{uid}.mp4"

    if direct.exists():
        return direct

    matches = list(VIDEO_ROOT.glob(f"{uid}*.mp4"))

    if len(matches) == 1:
        return matches[0]

    if len(matches) > 1:

        # Prefer exact filename if present.
        exact = [
            p for p in matches
            if p.stem == uid
        ]

        if len(exact) == 1:
            return exact[0]

        raise RuntimeError(
            f"Multiple video files found for UID {uid}:\n"
            + "\n".join(str(p) for p in matches)
        )

    return None


# =============================================================================
# 8. ROBUST SEQUENTIAL VIDEO DECODER
# =============================================================================
#
# IMPORTANT DIFFERENCE FROM THE PREVIOUS VERSION:
#
# We DO NOT do:
#
#     cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
#
# because some MP4 files have inaccurate frame indexes / damaged frames.
#
# Instead:
#
#     frame 0 -> frame 1 -> frame 2 -> ... -> EOF/failure
#
# Then we sample uniformly from the successfully decoded sequence.
#
# =============================================================================


def get_uniform_frame_indices(actual_frame_count, num_frames=8):

    if actual_frame_count <= 0:
        raise ValueError(
            "No successfully decoded frames."
        )

    if actual_frame_count == 1:
        return np.zeros(
            num_frames,
            dtype=np.int64
        )

    indices = np.linspace(
        0,
        actual_frame_count - 1,
        num_frames
    )

    return np.round(indices).astype(np.int64)


def read_uniform_frames(
    video_path,
    num_frames=8,
    return_metadata=False
):
    """
    Robust sequential decoder.

    Returns:
        frames:
            uint8 RGB array
            shape = (num_frames, 224, 224, 3)

        metadata:
            decoding diagnostics
    """

    video_path = Path(video_path)

    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        raise RuntimeError(
            f"Could not open video:\n{video_path}"
        )

    reported_frame_count = int(
        cap.get(cv2.CAP_PROP_FRAME_COUNT)
    )

    reported_fps = float(
        cap.get(cv2.CAP_PROP_FPS)
    )

    decoded_frames = []

    decode_failure = False
    failure_position = None

    while True:

        success, frame = cap.read()

        if not success:

            decode_failure = True
            failure_position = len(decoded_frames)

            break

        frame = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        frame = cv2.resize(
            frame,
            (IMAGE_SIZE, IMAGE_SIZE),
            interpolation=cv2.INTER_LINEAR
        )

        decoded_frames.append(frame)

    cap.release()

    actual_frame_count = len(decoded_frames)

    if actual_frame_count == 0:
        raise RuntimeError(
            f"No successfully decoded frames:\n{video_path}"
        )

    sample_indices = get_uniform_frame_indices(
        actual_frame_count,
        num_frames
    )

    sampled_frames = [
        decoded_frames[int(idx)]
        for idx in sample_indices
    ]

    frames = np.stack(
        sampled_frames,
        axis=0
    )

    metadata = {
        "video_path": str(video_path),
        "reported_frame_count": int(
            reported_frame_count
        ),
        "actual_decoded_frame_count": int(
            actual_frame_count
        ),
        "reported_fps": float(
            reported_fps
        ),
        "sample_indices": sample_indices.tolist(),
        "decode_truncated": bool(
            reported_frame_count > 0
            and actual_frame_count < reported_frame_count
        ),
        "decode_failure": bool(
            decode_failure
        ),
        "decode_failure_position": (
            int(failure_position)
            if failure_position is not None
            else None
        ),
    }

    if (
        reported_frame_count > 0
        and actual_frame_count < reported_frame_count
    ):

        print(
            f"[ROBUST DECODE] "
            f"{video_path.name} | "
            f"reported={reported_frame_count} | "
            f"decoded={actual_frame_count} | "
            f"using actual decoded frames"
        )

    if return_metadata:
        return frames, metadata

    return frames


# =============================================================================
# 9. TEST ROBUST DECODER ON PREVIOUSLY PROBLEMATIC FILE
# =============================================================================

problem_video = (
    VIDEO_ROOT /
    "-0XujLX2_9I_2.mp4"
)

if problem_video.exists():

    print("\nTesting robust decoder on previously problematic video:")
    print(problem_video)

    test_frames, test_metadata = read_uniform_frames(
        problem_video,
        num_frames=NUM_FRAMES,
        return_metadata=True
    )

    assert test_frames.shape == (
        NUM_FRAMES,
        IMAGE_SIZE,
        IMAGE_SIZE,
        3
    )

    assert test_frames.dtype == np.uint8

    print("\n[ROBUST DECODER TEST PASSED]")
    print(json.dumps(
        test_metadata,
        indent=2
    ))

else:
    print(
        "\nPreviously problematic video not found; "
        "continuing with general decoder."
    )


# =============================================================================
# 10. PREPROCESSING
# =============================================================================

eval_transform = transforms.Compose([
    transforms.ToPILImage(),

    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])


# =============================================================================
# 11. MODEL CHECKPOINT PATHS
# =============================================================================

MODEL_PATHS = {

    "ResNet18":
        MODEL_ROOT /
        "resnet18_stage4_4_v2_best.pt",

    "MobileNetV3-Large":
        MODEL_ROOT /
        "mobilenet_v3_large_v2" /
        "mobilenet_v3_large_stage4_4_v2_best.pt",

    "EfficientNet-B0":
        MODEL_ROOT /
        "efficientnet_b0_v2_exact_meanlogit" /
        "efficientnet_b0_stage4_4_v2_exact_meanlogit_best.pt",
}


# =============================================================================
# 12. MODEL BUILDERS
# =============================================================================

def build_resnet18():

    model = models.resnet18(
        weights=None
    )

    model.fc = nn.Linear(
        model.fc.in_features,
        NUM_CLASSES
    )

    return model


def build_mobilenet_v3_large():

    model = models.mobilenet_v3_large(
        weights=None
    )

    model.classifier[3] = nn.Linear(
        model.classifier[3].in_features,
        NUM_CLASSES
    )

    return model


def build_efficientnet_b0():

    model = models.efficientnet_b0(
        weights=None
    )

    model.classifier[1] = nn.Linear(
        model.classifier[1].in_features,
        NUM_CLASSES
    )

    return model


# =============================================================================
# 13. CHECKPOINT LOADER
# =============================================================================

def clean_state_dict_keys(
    state_dict,
    model_name
):
    """
    Handles the EfficientNet Stage 4.4 training wrapper
    where checkpoint keys may contain:

        features.features.*

    while torchvision EfficientNet expects:

        features.*
    """

    cleaned = {}

    for key, value in state_dict.items():

        new_key = key

        if model_name == "EfficientNet-B0":

            if new_key.startswith(
                "features.features."
            ):

                new_key = new_key.replace(
                    "features.features.",
                    "features.",
                    1
                )

        cleaned[new_key] = value

    return cleaned


def load_checkpoint(
    model,
    checkpoint_path,
    model_name
):

    checkpoint_path = Path(
        checkpoint_path
    )

    print(
        f"\nLoading {model_name}:"
    )

    print(
        checkpoint_path
    )

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu"
    )

    # ---------------------------------------------------------
    # Detect checkpoint structure
    # ---------------------------------------------------------

    if isinstance(checkpoint, dict):

        if "model_state_dict" in checkpoint:

            state_dict = checkpoint[
                "model_state_dict"
            ]

        elif "state_dict" in checkpoint:

            state_dict = checkpoint[
                "state_dict"
            ]

        else:

            # checkpoint itself is probably state_dict
            state_dict = checkpoint

    else:

        raise RuntimeError(
            f"Unsupported checkpoint format:\n"
            f"{checkpoint_path}"
        )

    # ---------------------------------------------------------
    # Remove DataParallel prefix
    # ---------------------------------------------------------

    cleaned_state_dict = {}

    for key, value in state_dict.items():

        if key.startswith(
            "module."
        ):

            key = key[
                len("module.") :
            ]

        cleaned_state_dict[key] = value

    # ---------------------------------------------------------
    # Model-specific key cleanup
    # ---------------------------------------------------------

    cleaned_state_dict = clean_state_dict_keys(
        cleaned_state_dict,
        model_name
    )

    # ---------------------------------------------------------
    # Load
    # ---------------------------------------------------------

    missing_keys, unexpected_keys = model.load_state_dict(
        cleaned_state_dict,
        strict=False
    )

    if len(missing_keys) > 0:

        print(
            "\nWARNING — Missing checkpoint keys:"
        )

        for key in missing_keys[:20]:
            print(
                "   ",
                key
            )

    if len(unexpected_keys) > 0:

        print(
            "\nWARNING — Unexpected checkpoint keys:"
        )

        for key in unexpected_keys[:20]:
            print(
                "   ",
                key
            )

    if len(missing_keys) == 0 and len(unexpected_keys) == 0:

        print(
            "[OK] Checkpoint loaded with no key mismatch."
        )

    model = model.to(DEVICE)
    model.eval()

    return model


# =============================================================================
# 14. BUILD ALL MODELS
# =============================================================================

print("\n" + "=" * 80)
print("BUILDING MODELS")
print("=" * 80)

models_dict = {}


# -----------------------------------------------------------------------------
# ResNet18
# -----------------------------------------------------------------------------

resnet18_model = build_resnet18()

resnet18_model = load_checkpoint(
    resnet18_model,
    MODEL_PATHS["ResNet18"],
    "ResNet18"
)

models_dict["ResNet18"] = resnet18_model


# -----------------------------------------------------------------------------
# MobileNetV3-Large
# -----------------------------------------------------------------------------

mobilenet_model = build_mobilenet_v3_large()

mobilenet_model = load_checkpoint(
    mobilenet_model,
    MODEL_PATHS["MobileNetV3-Large"],
    "MobileNetV3-Large"
)

models_dict["MobileNetV3-Large"] = mobilenet_model


# -----------------------------------------------------------------------------
# EfficientNet-B0
# -----------------------------------------------------------------------------

efficientnet_model = build_efficientnet_b0()

efficientnet_model = load_checkpoint(
    efficientnet_model,
    MODEL_PATHS["EfficientNet-B0"],
    "EfficientNet-B0"
)

models_dict["EfficientNet-B0"] = efficientnet_model


print("\n[OK] All three Stage 4.4 models loaded.")


# =============================================================================
# 15. PENULTIMATE EMBEDDING EXTRACTORS
# =============================================================================
#
# We DO NOT use the 4764-dimensional classifier logits.
#
# Instead:
#
# ResNet18:
#     backbone -> avgpool -> 512-d embedding
#
# MobileNetV3-Large:
#     features -> avgpool -> 1280-d embedding
#
# EfficientNet-B0:
#     features -> avgpool -> 1280-d embedding
#
# =============================================================================


@torch.no_grad()
def extract_resnet18_embedding(
    model,
    frame_batch
):

    x = model.conv1(frame_batch)
    x = model.bn1(x)
    x = model.relu(x)
    x = model.maxpool(x)

    x = model.layer1(x)
    x = model.layer2(x)
    x = model.layer3(x)
    x = model.layer4(x)

    x = model.avgpool(x)

    x = torch.flatten(
        x,
        1
    )

    return x


@torch.no_grad()
def extract_mobilenet_embedding(
    model,
    frame_batch
):

    x = model.features(
        frame_batch
    )

    x = model.avgpool(
        x
    )

    x = torch.flatten(
        x,
        1
    )

    return x


@torch.no_grad()
def extract_efficientnet_embedding(
    model,
    frame_batch
):

    x = model.features(
        frame_batch
    )

    x = model.avgpool(
        x
    )

    x = torch.flatten(
        x,
        1
    )

    return x


def extract_frame_embeddings(
    model_name,
    model,
    frames
):

    # ---------------------------------------------------------
    # Convert uint8 RGB frames -> normalized tensors
    # ---------------------------------------------------------

    tensors = []

    for frame in frames:

        tensor = eval_transform(
            frame
        )

        tensors.append(
            tensor
        )

    x = torch.stack(
        tensors,
        dim=0
    )

    x = x.to(
        DEVICE,
        non_blocking=True
    )

    # ---------------------------------------------------------
    # Extract embedding for each frame
    # ---------------------------------------------------------

    if model_name == "ResNet18":

        embeddings = extract_resnet18_embedding(
            model,
            x
        )

    elif model_name == "MobileNetV3-Large":

        embeddings = extract_mobilenet_embedding(
            model,
            x
        )

    elif model_name == "EfficientNet-B0":

        embeddings = extract_efficientnet_embedding(
            model,
            x
        )

    else:

        raise ValueError(
            f"Unknown model: {model_name}"
        )

    # ---------------------------------------------------------
    # Mean across 8 frames
    # ---------------------------------------------------------

    video_embedding = embeddings.mean(
        dim=0
    )

    # ---------------------------------------------------------
    # L2 normalization
    # ---------------------------------------------------------

    video_embedding = F.normalize(
        video_embedding,
        p=2,
        dim=0
    )

    return video_embedding.cpu().numpy().astype(
        np.float32
    )


# =============================================================================
# 16. PREPARE OFFICIAL PROTOTYPE / TEST DATA
# =============================================================================

def prepare_split_dataframe(
    df,
    split_name
):

    rows = []

    missing_video_paths = []

    invalid_gloss = []

    for _, row in df.iterrows():

        uid = str(
            row["uid"]
        )

        gloss = row["gloss"]

        if pd.isna(gloss):

            invalid_gloss.append(
                uid
            )

            continue

        gloss = str(
            gloss
        )

        if gloss not in gloss_to_class:

            invalid_gloss.append(
                uid
            )

            continue

        video_path = resolve_video_path(
            uid
        )

        if video_path is None:

            missing_video_paths.append(
                uid
            )

            continue

        class_id = gloss_to_class[
            gloss
        ]

        rows.append({

            "uid":
                uid,

            "gloss":
                gloss,

            "class_id":
                int(class_id),

            "video_path":
                str(video_path),

            "split":
                split_name,
        })

    result = pd.DataFrame(
        rows
    )

    print(
        f"\n{split_name} preparation:"
    )

    print(
        f"  valid rows       : {len(result)}"
    )

    print(
        f"  missing videos   : {len(missing_video_paths)}"
    )

    print(
        f"  invalid glosses  : {len(invalid_gloss)}"
    )

    if len(missing_video_paths) > 0:

        print(
            "\nMissing video UIDs:"
        )

        for uid in missing_video_paths[:20]:
            print(
                "  ",
                uid
            )

    if len(invalid_gloss) > 0:

        print(
            "\nInvalid gloss UIDs:"
        )

        for uid in invalid_gloss[:20]:
            print(
                "  ",
                uid
            )

    return result


prototype_eval_df = prepare_split_dataframe(
    prototype_df,
    "prototype"
)

test_eval_df = prepare_split_dataframe(
    test_df,
    "test"
)


# =============================================================================
# 17. VERIFY OFFICIAL PROTOTYPE REQUIREMENT
# =============================================================================
#
# CISLR official setup:
#
#   one prototype/support video per class
#
# =============================================================================

prototype_counts = (
    prototype_eval_df
    .groupby("class_id")
    .size()
)

print(
    "\nPrototype class statistics:"
)

print(
    f"  classes represented : {prototype_counts.shape[0]}"
)

print(
    f"  min videos/class    : {prototype_counts.min()}"
)

print(
    f"  max videos/class    : {prototype_counts.max()}"
)

if prototype_counts.shape[0] != NUM_CLASSES:

    raise RuntimeError(
        f"Expected {NUM_CLASSES} prototype classes, "
        f"found {prototype_counts.shape[0]}."
    )

if not np.all(
    prototype_counts.values == 1
):

    raise RuntimeError(
        "Prototype split does not contain exactly "
        "one prototype video per class."
    )

print(
    "[OK] Exactly one prototype per class."
)


# =============================================================================
# 18. VIDEO EMBEDDING EXTRACTION FUNCTION
# =============================================================================

def extract_split_embeddings(
    model_name,
    model,
    split_df,
    split_name
):

    print("\n" + "=" * 80)

    print(
        f"EXTRACTING {model_name} — {split_name.upper()}"
    )

    print("=" * 80)

    embeddings = []

    metadata_rows = []

    failed_rows = []

    start_time = time.time()

    for _, row in tqdm(
        split_df.iterrows(),
        total=len(split_df),
        desc=f"{model_name} / {split_name}"
    ):

        uid = str(
            row["uid"]
        )

        video_path = Path(
            row["video_path"]
        )

        try:

            frames, decode_metadata = read_uniform_frames(
                video_path,
                num_frames=NUM_FRAMES,
                return_metadata=True
            )

            embedding = extract_frame_embeddings(
                model_name,
                model,
                frames
            )

            embeddings.append(
                embedding
            )

            metadata_rows.append({

                "uid":
                    uid,

                "gloss":
                    row["gloss"],

                "class_id":
                    int(row["class_id"]),

                "video_path":
                    str(video_path),

                "split":
                    split_name,

                "reported_frame_count":
                    decode_metadata[
                        "reported_frame_count"
                    ],

                "actual_decoded_frame_count":
                    decode_metadata[
                        "actual_decoded_frame_count"
                    ],

                "reported_fps":
                    decode_metadata[
                        "reported_fps"
                    ],

                "sample_indices":
                    json.dumps(
                        decode_metadata[
                            "sample_indices"
                        ]
                    ),

                "decode_truncated":
                    decode_metadata[
                        "decode_truncated"
                    ],

                "decode_failure":
                    decode_metadata[
                        "decode_failure"
                    ],

                "decode_failure_position":
                    decode_metadata[
                        "decode_failure_position"
                    ],

                "status":
                    "OK",

                "error":
                    "",
            })

        except Exception as e:

            print(
                f"\n[ERROR] {uid}"
            )

            print(
                str(e)
            )

            failed_rows.append({

                "uid":
                    uid,

                "gloss":
                    row["gloss"],

                "class_id":
                    int(row["class_id"]),

                "video_path":
                    str(video_path),

                "split":
                    split_name,

                "status":
                    "FAILED",

                "error":
                    str(e),
            })

    elapsed = (
        time.time()
        - start_time
    )

    # -------------------------------------------------------------------------
    # Validate
    # -------------------------------------------------------------------------

    if len(failed_rows) > 0:

        failed_df = pd.DataFrame(
            failed_rows
        )

        failure_path = (
            STAGE45_ROOT /
            f"{model_name.replace('/', '_')}_{split_name}_failures.csv"
        )

        failed_df.to_csv(
            failure_path,
            index=False
        )

        print(
            f"\nFAILED videos: {len(failed_rows)}"
        )

        print(
            f"Failure audit saved to:\n"
            f"{failure_path}"
        )

        raise RuntimeError(
            f"{len(failed_rows)} {split_name} videos "
            f"could not be evaluated. "
            f"Do not silently exclude them."
        )

    embeddings = np.stack(
        embeddings,
        axis=0
    ).astype(
        np.float32
    )

    metadata_df = pd.DataFrame(
        metadata_rows
    )

    print(
        f"\nEmbedding shape: {embeddings.shape}"
    )

    print(
        f"Extraction time: {elapsed / 60:.2f} min"
    )

    return embeddings, metadata_df


# =============================================================================
# 19. OFFICIAL RETRIEVAL METRICS
# =============================================================================

def compute_retrieval_metrics(
    similarity_matrix,
    prototype_class_ids,
    test_class_ids
):

    num_queries = similarity_matrix.shape[0]

    ranks = []

    top1_hits = 0
    top5_hits = 0
    recall10_hits = 0

    reciprocal_ranks = []

    top1_predictions = []
    top5_predictions = []

    for i in range(num_queries):

        similarities = similarity_matrix[i]

        ranking = np.argsort(
            -similarities
        )

        ranked_classes = np.array(
            prototype_class_ids
        )[ranking]

        true_class = int(
            test_class_ids[i]
        )

        matches = np.where(
            ranked_classes == true_class
        )[0]

        if len(matches) == 0:

            raise RuntimeError(
                f"True class {true_class} has no prototype."
            )

        rank = int(
            matches[0]
        ) + 1

        ranks.append(
            rank
        )

        # Top-1
        if rank <= 1:
            top1_hits += 1

        # Top-5
        if rank <= 5:
            top5_hits += 1

        # Recall@10
        if rank <= 10:
            recall10_hits += 1

        reciprocal_ranks.append(
            1.0 / rank
        )

        top1_predictions.append(
            int(ranked_classes[0])
        )

        top5_predictions.append(
            [
                int(x)
                for x in ranked_classes[:5]
            ]
        )

    ranks = np.asarray(
        ranks,
        dtype=np.int32
    )

    metrics = {

        "num_queries":
            int(num_queries),

        "num_prototypes":
            int(similarity_matrix.shape[1]),

        "Top-1":
            float(
                top1_hits / num_queries
            ),

        "Top-5":
            float(
                top5_hits / num_queries
            ),

        "Recall@10":
            float(
                recall10_hits / num_queries
            ),

        "MRR":
            float(
                np.mean(
                    reciprocal_ranks
                )
            ),

        "mean_rank":
            float(
                np.mean(ranks)
            ),

        "median_rank":
            float(
                np.median(ranks)
            ),

        "min_rank":
            int(
                np.min(ranks)
            ),

        "max_rank":
            int(
                np.max(ranks)
            ),
    }

    return (
        metrics,
        ranks,
        top1_predictions,
        top5_predictions
    )


# =============================================================================
# 20. RUN OFFICIAL RETRIEVAL
# =============================================================================

all_results = {}

decode_audits = []


for model_name, model in models_dict.items():

    print("\n\n")
    print("#" * 80)
    print(
        f"# OFFICIAL EVALUATION — {model_name}"
    )
    print("#" * 80)

    # -------------------------------------------------------------------------
    # Prototype embeddings
    # -------------------------------------------------------------------------

    prototype_embeddings, prototype_metadata = (
        extract_split_embeddings(
            model_name,
            model,
            prototype_eval_df,
            "prototype"
        )
    )

    # -------------------------------------------------------------------------
    # Test embeddings
    # -------------------------------------------------------------------------

    test_embeddings, test_metadata = (
        extract_split_embeddings(
            model_name,
            model,
            test_eval_df,
            "test"
        )
    )

    # -------------------------------------------------------------------------
    # Save embeddings
    # -------------------------------------------------------------------------

    model_slug = (
        model_name
        .lower()
        .replace("-", "_")
        .replace(" ", "_")
    )

    prototype_embedding_path = (
        EMBEDDING_ROOT /
        f"{model_slug}_prototype_embeddings.npy"
    )

    test_embedding_path = (
        EMBEDDING_ROOT /
        f"{model_slug}_test_embeddings.npy"
    )

    np.save(
        prototype_embedding_path,
        prototype_embeddings
    )

    np.save(
        test_embedding_path,
        test_embeddings
    )

    prototype_metadata_path = (
        EMBEDDING_ROOT /
        f"{model_slug}_prototype_metadata.csv"
    )

    test_metadata_path = (
        EMBEDDING_ROOT /
        f"{model_slug}_test_metadata.csv"
    )

    prototype_metadata.to_csv(
        prototype_metadata_path,
        index=False
    )

    test_metadata.to_csv(
        test_metadata_path,
        index=False
    )

    # -------------------------------------------------------------------------
    # Save decode audits
    # -------------------------------------------------------------------------

    prototype_audit = prototype_metadata.copy()

    prototype_audit.insert(
        0,
        "model",
        model_name
    )

    test_audit = test_metadata.copy()

    test_audit.insert(
        0,
        "model",
        model_name
    )

    decode_audits.append(
        prototype_audit
    )

    decode_audits.append(
        test_audit
    )

    # -------------------------------------------------------------------------
    # Verify embedding dimensions
    # -------------------------------------------------------------------------

    if model_name == "ResNet18":

        expected_dim = 512

    else:

        expected_dim = 1280

    if prototype_embeddings.shape[1] != expected_dim:

        raise RuntimeError(
            f"{model_name}: expected embedding dimension "
            f"{expected_dim}, got "
            f"{prototype_embeddings.shape[1]}"
        )

    if test_embeddings.shape[1] != expected_dim:

        raise RuntimeError(
            f"{model_name}: expected embedding dimension "
            f"{expected_dim}, got "
            f"{test_embeddings.shape[1]}"
        )

    # -------------------------------------------------------------------------
    # Ensure normalized embeddings
    # -------------------------------------------------------------------------

    prototype_norms = np.linalg.norm(
        prototype_embeddings,
        axis=1
    )

    test_norms = np.linalg.norm(
        test_embeddings,
        axis=1
    )

    if not np.allclose(
        prototype_norms,
        1.0,
        atol=1e-4
    ):

        raise RuntimeError(
            f"{model_name}: prototype embeddings "
            f"are not L2 normalized."
        )

    if not np.allclose(
        test_norms,
        1.0,
        atol=1e-4
    ):

        raise RuntimeError(
            f"{model_name}: test embeddings "
            f"are not L2 normalized."
        )

    # -------------------------------------------------------------------------
    # Prototype ordering
    # -------------------------------------------------------------------------

    prototype_class_ids = (
        prototype_metadata[
            "class_id"
        ]
        .to_numpy()
        .astype(np.int32)
    )

    test_class_ids = (
        test_metadata[
            "class_id"
        ]
        .to_numpy()
        .astype(np.int32)
    )

    # -------------------------------------------------------------------------
    # Cosine similarity
    #
    # Since embeddings are already L2-normalized:
    #
    #       cosine(a,b) = a dot b
    #
    # -------------------------------------------------------------------------

    print(
        "\nComputing prototype -> test cosine similarity..."
    )

    similarity_matrix = (
        test_embeddings
        @ prototype_embeddings.T
    ).astype(
        np.float32
    )

    print(
        f"Similarity matrix: "
        f"{similarity_matrix.shape}"
    )

    # -------------------------------------------------------------------------
    # Save similarity matrix
    # -------------------------------------------------------------------------

    similarity_path = (
        SIMILARITY_ROOT /
        f"{model_slug}_test_vs_prototype_cosine.npy"
    )

    np.save(
        similarity_path,
        similarity_matrix
    )

    # -------------------------------------------------------------------------
    # Metrics
    # -------------------------------------------------------------------------

    (
        metrics,
        ranks,
        top1_predictions,
        top5_predictions
    ) = compute_retrieval_metrics(
        similarity_matrix,
        prototype_class_ids,
        test_class_ids
    )

    metrics["model"] = model_name
    metrics["embedding_dimension"] = int(
        prototype_embeddings.shape[1]
    )

    metrics["num_prototype_videos"] = int(
        len(prototype_metadata)
    )

    metrics["num_test_videos"] = int(
        len(test_metadata)
    )

    # -------------------------------------------------------------------------
    # Per-query prediction table
    # -------------------------------------------------------------------------

    prediction_rows = []

    for i in range(
        len(test_metadata)
    ):

        ranking = np.argsort(
            -similarity_matrix[i]
        )

        ranked_classes = np.array(
            prototype_class_ids
        )[ranking]

        ranked_scores = similarity_matrix[
            i,
            ranking
        ]

        true_class = int(
            test_class_ids[i]
        )

        true_rank = int(
            ranks[i]
        )

        prediction_rows.append({

            "uid":
                test_metadata.iloc[i]["uid"],

            "gloss":
                test_metadata.iloc[i]["gloss"],

            "true_class_id":
                true_class,

            "true_rank":
                true_rank,

            "top1_class_id":
                int(
                    ranked_classes[0]
                ),

            "top1_gloss":
                class_to_gloss[
                    int(
                        ranked_classes[0]
                    )
                ],

            "top1_similarity":
                float(
                    ranked_scores[0]
                ),

            "top5_class_ids":
                json.dumps(
                    [
                        int(x)
                        for x in ranked_classes[:5]
                    ]
                ),

            "top5_glosses":
                json.dumps(
                    [
                        class_to_gloss[
                            int(x)
                        ]
                        for x in ranked_classes[:5]
                    ],
                    ensure_ascii=False
                ),

            "top5_similarities":
                json.dumps(
                    [
                        float(x)
                        for x in ranked_scores[:5]
                    ]
                ),

            "hit_top1":
                bool(
                    true_rank <= 1
                ),

            "hit_top5":
                bool(
                    true_rank <= 5
                ),

            "hit_top10":
                bool(
                    true_rank <= 10
                ),
        })

    prediction_df = pd.DataFrame(
        prediction_rows
    )

    prediction_path = (
        PREDICTION_ROOT /
        f"{model_slug}_official_predictions.csv"
    )

    prediction_df.to_csv(
        prediction_path,
        index=False
    )

    # -------------------------------------------------------------------------
    # Save rank array
    # -------------------------------------------------------------------------

    rank_path = (
        PREDICTION_ROOT /
        f"{model_slug}_ranks.npy"
    )

    np.save(
        rank_path,
        ranks
    )

    # -------------------------------------------------------------------------
    # Save metrics
    # -------------------------------------------------------------------------

    metrics_path = (
        METRIC_ROOT /
        f"{model_slug}_official_metrics.json"
    )

    with open(
        metrics_path,
        "w"
    ) as f:

        json.dump(
            metrics,
            f,
            indent=2
        )

    # -------------------------------------------------------------------------
    # Store result
    # -------------------------------------------------------------------------

    all_results[
        model_name
    ] = metrics

    print("\n" + "-" * 70)

    print(
        f"RESULTS — {model_name}"
    )

    print("-" * 70)

    print(
        f"Top-1      : {metrics['Top-1']:.4f}"
    )

    print(
        f"Top-5      : {metrics['Top-5']:.4f}"
    )

    print(
        f"Recall@10  : {metrics['Recall@10']:.4f}"
    )

    print(
        f"MRR        : {metrics['MRR']:.4f}"
    )

    print(
        f"Mean Rank  : {metrics['mean_rank']:.2f}"
    )

    print(
        f"Median Rank: {metrics['median_rank']:.2f}"
    )

    print("-" * 70)


# =============================================================================
# 21. SAVE COMPLETE DECODE AUDIT
# =============================================================================

print("\n" + "=" * 80)
print("SAVING COMPLETE DECODE AUDIT")
print("=" * 80)

decode_audit_df = pd.concat(
    decode_audits,
    ignore_index=True
)

decode_audit_path = (
    STAGE45_ROOT /
    "stage4_5_decode_audit.csv"
)

decode_audit_df.to_csv(
    decode_audit_path,
    index=False
)

print(
    f"Decode audit saved:\n"
    f"{decode_audit_path}"
)


# =============================================================================
# 22. DECODE AUDIT SUMMARY
# =============================================================================

print("\n" + "=" * 80)
print("DECODE AUDIT SUMMARY")
print("=" * 80)

for model_name in models_dict.keys():

    model_audit = decode_audit_df[
        decode_audit_df["model"] == model_name
    ]

    truncated = model_audit[
        model_audit["decode_truncated"] == True
    ]

    failed = model_audit[
        model_audit["decode_failure"] == True
    ]

    print(
        f"\n{model_name}"
    )

    print(
        f"  total videos       : {len(model_audit)}"
    )

    print(
        f"  truncated videos   : {len(truncated)}"
    )

    print(
        f"  decode failures    : {len(failed)}"
    )

    if len(truncated) > 0:

        print(
            "  Example truncated videos:"
        )

        for _, row in truncated.head(10).iterrows():

            print(
                f"    {Path(row['video_path']).name} | "
                f"reported={row['reported_frame_count']} | "
                f"decoded={row['actual_decoded_frame_count']}"
            )


# =============================================================================
# 23. FINAL MODEL COMPARISON
# =============================================================================

comparison_rows = []

for model_name, metrics in all_results.items():

    comparison_rows.append({

        "model":
            model_name,

        "embedding_dimension":
            metrics[
                "embedding_dimension"
            ],

        "num_prototype_videos":
            metrics[
                "num_prototype_videos"
            ],

        "num_test_videos":
            metrics[
                "num_test_videos"
            ],

        "Top-1":
            metrics[
                "Top-1"
            ],

        "Top-5":
            metrics[
                "Top-5"
            ],

        "Recall@10":
            metrics[
                "Recall@10"
            ],

        "MRR":
            metrics[
                "MRR"
            ],

        "mean_rank":
            metrics[
                "mean_rank"
            ],

        "median_rank":
            metrics[
                "median_rank"
            ],
    })


comparison_df = pd.DataFrame(
    comparison_rows
)

comparison_df = comparison_df.sort_values(
    by="Top-1",
    ascending=False
).reset_index(
    drop=True
)


# =============================================================================
# 24. SAVE FINAL COMPARISON
# =============================================================================

comparison_csv_path = (
    STAGE45_ROOT /
    "stage4_5_official_model_comparison.csv"
)

comparison_df.to_csv(
    comparison_csv_path,
    index=False
)


comparison_json_path = (
    STAGE45_ROOT /
    "stage4_5_official_model_comparison.json"
)

with open(
    comparison_json_path,
    "w"
) as f:

    json.dump(
        all_results,
        f,
        indent=2
    )


# =============================================================================
# 25. FINAL REPORT
# =============================================================================

print("\n\n")
print("#" * 80)
print("# STAGE 4.5 — OFFICIAL CISLR EVALUATION COMPLETE")
print("#" * 80)

print("\nOfficial retrieval protocol:")
print("  Prototype videos : 4764")
print("  Test videos      : 2285")
print("  Prototype/test overlap : 0")
print("  Frames/video     : 8")
print("  Frame selection  : uniform over successfully decoded frames")
print("  Representation   : penultimate backbone embedding")
print("  Aggregation      : mean over 8 frame embeddings")
print("  Normalization    : L2")
print("  Similarity       : cosine similarity")
print("  Retrieval        : prototype -> test")
print("  Metrics          : Top-1, Top-5, Recall@10, MRR, Mean Rank, Median Rank")

print("\n" + "=" * 80)
print("FINAL MODEL COMPARISON")
print("=" * 80)

display_df = comparison_df.copy()

display_df["Top-1"] = (
    display_df["Top-1"] * 100
).round(2)

display_df["Top-5"] = (
    display_df["Top-5"] * 100
).round(2)

display_df["Recall@10"] = (
    display_df["Recall@10"] * 100
).round(2)

display_df["MRR"] = (
    display_df["MRR"]
).round(4)

display_df["mean_rank"] = (
    display_df["mean_rank"]
).round(2)

display_df["median_rank"] = (
    display_df["median_rank"]
).round(2)

print(
    display_df.to_string(
        index=False
    )
)


# =============================================================================
# 26. SAVE STAGE 4.5 PROTOCOL
# =============================================================================

stage45_protocol = {

    "stage":
        "4.5",

    "title":
        "Official CISLR Prototype-to-Test Retrieval Evaluation",

    "dataset":
        "CISLR v1.5-a",

    "num_classes":
        NUM_CLASSES,

    "prototype_videos":
        int(len(prototype_eval_df)),

    "test_videos":
        int(len(test_eval_df)),

    "prototype_test_uid_overlap":
        0,

    "frames_per_video":
        NUM_FRAMES,

    "image_size":
        IMAGE_SIZE,

    "frame_decoding":
        {
            "method":
                "sequential OpenCV decoding",

            "direct_frame_seeking":
                False,

            "truncated_video_policy":
                "uniformly sample from successfully decoded frames",

            "raw_video_modified":
                False,

            "zero_decodable_frames":
                "evaluation failure"
        },

    "preprocessing":
        {
            "color":
                "RGB",

            "resize":
                "224x224",

            "normalization":
                "ImageNet mean/std",

            "augmentation":
                False
        },

    "representation":
        {
            "type":
                "penultimate backbone embedding",

            "frame_aggregation":
                "mean over 8 frame embeddings",

            "normalization":
                "L2"
        },

    "similarity":
        "cosine similarity",

    "evaluation":
        "prototype-to-test retrieval",

    "metrics":
        [
            "Top-1",
            "Top-5",
            "Recall@10",
            "MRR",
            "mean_rank",
            "median_rank"
        ],

    "models":
        list(
            models_dict.keys()
        ),

    "checkpoint_paths":
        {
            name:
                str(path)
            for name, path
            in MODEL_PATHS.items()
        },

    "seed":
        SEED,
}


protocol_path = (
    STAGE45_ROOT /
    "stage4_5_protocol.json"
)

with open(
    protocol_path,
    "w"
) as f:

    json.dump(
        stage45_protocol,
        f,
        indent=2
    )


# =============================================================================
# 27. FINAL ARTIFACT LIST
# =============================================================================

print("\n" + "=" * 80)
print("STAGE 4.5 ARTIFACTS")
print("=" * 80)

print(
    f"\nRoot:\n{STAGE45_ROOT}"
)

print(
    "\nEmbeddings:"
)

print(
    f"  {EMBEDDING_ROOT}"
)

print(
    "\nPredictions:"
)

print(
    f"  {PREDICTION_ROOT}"
)

print(
    "\nSimilarity matrices:"
)

print(
    f"  {SIMILARITY_ROOT}"
)

print(
    "\nMetrics:"
)

print(
    f"  {METRIC_ROOT}"
)

print(
    "\nDecode audit:"
)

print(
    f"  {decode_audit_path}"
)

print(
    "\nFinal comparison:"
)

print(
    f"  {comparison_csv_path}"
)

print(
    f"  {comparison_json_path}"
)

print(
    "\nProtocol:"
)

print(
    f"  {protocol_path}"
)

print("\n" + "=" * 80)
print("STAGE 4.5 FINISHED SUCCESSFULLY")
print("=" * 80)

STAGE 4.5 — OFFICIAL CISLR EVALUATION
Project root : /home/dipshikha/Music/cao
Video root   : /home/dipshikha/Music/cao/CISLR_v1.5-a_videos
Device       : cuda
Num classes  : 4764
Frames/video : 8
Image size   : 224

Checking required files...
[OK] /home/dipshikha/Music/cao/dataset.csv
[OK] /home/dipshikha/Music/cao/prototype.csv
[OK] /home/dipshikha/Music/cao/test.csv
[OK] /home/dipshikha/Music/cao/CISLR_RESEARCH/models/stage4_4/resnet18_stage4_4_v2_best.pt
[OK] /home/dipshikha/Music/cao/CISLR_RESEARCH/models/stage4_4/mobilenet_v3_large_v2/mobilenet_v3_large_stage4_4_v2_best.pt
[OK] /home/dipshikha/Music/cao/CISLR_RESEARCH/models/stage4_4/efficientnet_b0_v2_exact_meanlogit/efficientnet_b0_stage4_4_v2_exact_meanlogit_best.pt

All required files found.

Loading dataset metadata...
dataset.csv   : (7050, 4)
prototype.csv : (4765, 4)
test.csv      : (2285, 4)

After removing invalid/malformed UID:
dataset   : 7049
prototype : 4764
test      : 2285

[OK] Prototype/Test UID overlap = 0

Tes

ResNet18 / prototype:   0%|          | 0/4764 [00:00<?, ?it/s]

[ROBUST DECODE] -0XujLX2_9I_2.mp4 | reported=70 | decoded=58 | using actual decoded frames
[ROBUST DECODE] -2_NUxX4PVA_2.mp4 | reported=119 | decoded=107 | using actual decoded frames
[ROBUST DECODE] -5p4hn8J87A_2.mp4 | reported=78 | decoded=75 | using actual decoded frames
[ROBUST DECODE] -I0sjwzg0tY_2.mp4 | reported=197 | decoded=188 | using actual decoded frames
[ROBUST DECODE] 039-5dIML0I_2.mp4 | reported=103 | decoded=99 | using actual decoded frames
[ROBUST DECODE] 0ETSwMTLaac_2.mp4 | reported=98 | decoded=97 | using actual decoded frames
[ROBUST DECODE] 0F_szwrQs1g_2.mp4 | reported=87 | decoded=79 | using actual decoded frames
[ROBUST DECODE] 0JIsxA88oss_2.mp4 | reported=106 | decoded=100 | using actual decoded frames
[ROBUST DECODE] 0Qk8-_ZEmHM_2.mp4 | reported=96 | decoded=90 | using actual decoded frames
[ROBUST DECODE] 0fdRLmUIlZE_2.mp4 | reported=131 | decoded=125 | using actual decoded frames
[ROBUST DECODE] 0ijcVXYEvY8_2.mp4 | reported=112 | decoded=109 | using actual dec

ResNet18 / test:   0%|          | 0/2285 [00:00<?, ?it/s]

[ROBUST DECODE] 08E4Dr0zyKk_2.mp4 | reported=78 | decoded=69 | using actual decoded frames
[ROBUST DECODE] 4kKAKsQ5mGU_2.mp4 | reported=146 | decoded=139 | using actual decoded frames
[ROBUST DECODE] 8NqmODtSLgI_2.mp4 | reported=90 | decoded=84 | using actual decoded frames
[ROBUST DECODE] BSezNX5k4dA_2.mp4 | reported=76 | decoded=67 | using actual decoded frames
[ROBUST DECODE] CRdSfDtdSj8_2.mp4 | reported=148 | decoded=145 | using actual decoded frames
[ROBUST DECODE] CcX4eD-UArI_2.mp4 | reported=166 | decoded=160 | using actual decoded frames
[ROBUST DECODE] CjZKfMv3QsA_2.mp4 | reported=109 | decoded=102 | using actual decoded frames
[ROBUST DECODE] Dg1FJhcTg2E_2.mp4 | reported=87 | decoded=85 | using actual decoded frames
[ROBUST DECODE] DlyWuMwYeVc_2.mp4 | reported=147 | decoded=145 | using actual decoded frames
[ROBUST DECODE] EI4ConsbYws_2.mp4 | reported=123 | decoded=115 | using actual decoded frames
[ROBUST DECODE] EmUsRBX9664_2.mp4 | reported=95 | decoded=89 | using actual de

MobileNetV3-Large / prototype:   0%|          | 0/4764 [00:00<?, ?it/s]

[ROBUST DECODE] -0XujLX2_9I_2.mp4 | reported=70 | decoded=58 | using actual decoded frames
[ROBUST DECODE] -2_NUxX4PVA_2.mp4 | reported=119 | decoded=107 | using actual decoded frames
[ROBUST DECODE] -5p4hn8J87A_2.mp4 | reported=78 | decoded=75 | using actual decoded frames
[ROBUST DECODE] -I0sjwzg0tY_2.mp4 | reported=197 | decoded=188 | using actual decoded frames
[ROBUST DECODE] 039-5dIML0I_2.mp4 | reported=103 | decoded=99 | using actual decoded frames
[ROBUST DECODE] 0ETSwMTLaac_2.mp4 | reported=98 | decoded=97 | using actual decoded frames
[ROBUST DECODE] 0F_szwrQs1g_2.mp4 | reported=87 | decoded=79 | using actual decoded frames
[ROBUST DECODE] 0JIsxA88oss_2.mp4 | reported=106 | decoded=100 | using actual decoded frames
[ROBUST DECODE] 0Qk8-_ZEmHM_2.mp4 | reported=96 | decoded=90 | using actual decoded frames
[ROBUST DECODE] 0fdRLmUIlZE_2.mp4 | reported=131 | decoded=125 | using actual decoded frames
[ROBUST DECODE] 0ijcVXYEvY8_2.mp4 | reported=112 | decoded=109 | using actual dec

MobileNetV3-Large / test:   0%|          | 0/2285 [00:00<?, ?it/s]

[ROBUST DECODE] 08E4Dr0zyKk_2.mp4 | reported=78 | decoded=69 | using actual decoded frames
[ROBUST DECODE] 4kKAKsQ5mGU_2.mp4 | reported=146 | decoded=139 | using actual decoded frames
[ROBUST DECODE] 8NqmODtSLgI_2.mp4 | reported=90 | decoded=84 | using actual decoded frames
[ROBUST DECODE] BSezNX5k4dA_2.mp4 | reported=76 | decoded=67 | using actual decoded frames
[ROBUST DECODE] CRdSfDtdSj8_2.mp4 | reported=148 | decoded=145 | using actual decoded frames
[ROBUST DECODE] CcX4eD-UArI_2.mp4 | reported=166 | decoded=160 | using actual decoded frames
[ROBUST DECODE] CjZKfMv3QsA_2.mp4 | reported=109 | decoded=102 | using actual decoded frames
[ROBUST DECODE] Dg1FJhcTg2E_2.mp4 | reported=87 | decoded=85 | using actual decoded frames
[ROBUST DECODE] DlyWuMwYeVc_2.mp4 | reported=147 | decoded=145 | using actual decoded frames
[ROBUST DECODE] EI4ConsbYws_2.mp4 | reported=123 | decoded=115 | using actual decoded frames
[ROBUST DECODE] EmUsRBX9664_2.mp4 | reported=95 | decoded=89 | using actual de

RuntimeError: MobileNetV3-Large: expected embedding dimension 1280, got 960

In [3]:
# =============================================================================
# STAGE 4.5 — CONTINUATION CELL
# =============================================================================
#
# Previous cell stopped at:
#
#   MobileNetV3-Large: expected embedding dimension 1280, got 960
#
# IMPORTANT:
#   - ResNet18 has already been processed.
#   - DO NOT rerun ResNet18.
#   - This cell continues from MobileNetV3-Large.
#   - Then evaluates EfficientNet-B0.
#
# Correct embedding dimensions:
#   ResNet18          = 512
#   MobileNetV3-Large = 960
#   EfficientNet-B0   = 1280
#
# =============================================================================


print("=" * 80)
print("STAGE 4.5 — CONTINUATION")
print("=" * 80)

print("\nCorrect embedding dimensions:")
print("  ResNet18          : 512")
print("  MobileNetV3-Large : 960")
print("  EfficientNet-B0   : 1280")


# =============================================================================
# 1. VERIFY CURRENT NOTEBOOK STATE
# =============================================================================

required_variables = [
    "models_dict",
    "prototype_eval_df",
    "test_eval_df",
    "extract_split_embeddings",
    "compute_retrieval_metrics",
    "prototype_embeddings",
    "prototype_metadata",
    "model_name",
    "MODEL_ROOT",
    "EMBEDDING_ROOT",
    "PREDICTION_ROOT",
    "SIMILARITY_ROOT",
    "METRIC_ROOT",
    "STAGE45_ROOT",
    "class_to_gloss",
    "prototype_class_ids",
    "test_class_ids",
]

missing_variables = [
    name
    for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "The previous Stage 4.5 cell state is not available.\n"
        "Missing variables:\n"
        + "\n".join(
            f"  - {x}"
            for x in missing_variables
        )
    )

print("\n[OK] Previous Stage 4.5 state is available.")


# =============================================================================
# 2. MOBILE NET V3 LARGE
# =============================================================================
#
# The previous cell already performed:
#
#   prototype_embeddings, prototype_metadata =
#       extract_split_embeddings(...)
#
# So we DO NOT extract MobileNet prototypes again.
#
# =============================================================================

print("\n" + "=" * 80)
print("MOBILENETV3-LARGE — CONTINUING")
print("=" * 80)


model_name = "MobileNetV3-Large"

model = models_dict[
    model_name
]

expected_dim = 960

print(
    f"\nExisting MobileNet prototype embedding shape:"
)
print(
    prototype_embeddings.shape
)

if prototype_embeddings.shape[1] != expected_dim:

    raise RuntimeError(
        f"MobileNetV3-Large embedding dimension mismatch. "
        f"Expected {expected_dim}, got "
        f"{prototype_embeddings.shape[1]}"
    )

print(
    "[OK] MobileNetV3-Large embedding dimension = 960"
)


# =============================================================================
# 3. NORMALIZATION CHECK
# =============================================================================

prototype_norms = np.linalg.norm(
    prototype_embeddings,
    axis=1
)

if not np.allclose(
    prototype_norms,
    1.0,
    atol=1e-4
):

    print(
        "[WARNING] Prototype embeddings are not perfectly normalized."
    )

    prototype_embeddings = (
        prototype_embeddings /
        np.maximum(
            prototype_norms[:, None],
            1e-12
        )
    )

else:

    print(
        "[OK] MobileNet prototype embeddings are L2 normalized."
    )


# =============================================================================
# 4. SAVE MOBILENET PROTOTYPE EMBEDDINGS
# =============================================================================

model_slug = (
    model_name
    .lower()
    .replace("-", "_")
    .replace(" ", "_")
)

prototype_embedding_path = (
    EMBEDDING_ROOT /
    f"{model_slug}_prototype_embeddings.npy"
)

np.save(
    prototype_embedding_path,
    prototype_embeddings
)

prototype_metadata_path = (
    EMBEDDING_ROOT /
    f"{model_slug}_prototype_metadata.csv"
)

prototype_metadata.to_csv(
    prototype_metadata_path,
    index=False
)

print(
    "\nSaved MobileNet prototype embeddings:"
)

print(
    prototype_embedding_path
)


# =============================================================================
# 5. EXTRACT MOBILENET TEST EMBEDDINGS
# =============================================================================

print("\n" + "-" * 80)
print("Extracting MobileNetV3-Large TEST embeddings")
print("-" * 80)

test_embeddings, test_metadata = (
    extract_split_embeddings(
        model_name,
        model,
        test_eval_df,
        "test"
    )
)

print(
    "\nMobileNet test embedding shape:"
)

print(
    test_embeddings.shape
)

if test_embeddings.shape[1] != expected_dim:

    raise RuntimeError(
        f"MobileNetV3-Large test embedding dimension mismatch. "
        f"Expected {expected_dim}, got "
        f"{test_embeddings.shape[1]}"
    )


# =============================================================================
# 6. NORMALIZE TEST EMBEDDINGS
# =============================================================================

test_norms = np.linalg.norm(
    test_embeddings,
    axis=1
)

if not np.allclose(
    test_norms,
    1.0,
    atol=1e-4
):

    print(
        "[WARNING] Test embeddings are not perfectly normalized."
    )

    test_embeddings = (
        test_embeddings /
        np.maximum(
            test_norms[:, None],
            1e-12
        )
    )

else:

    print(
        "[OK] MobileNet test embeddings are L2 normalized."
    )


# =============================================================================
# 7. SAVE MOBILENET TEST EMBEDDINGS
# =============================================================================

test_embedding_path = (
    EMBEDDING_ROOT /
    f"{model_slug}_test_embeddings.npy"
)

np.save(
    test_embedding_path,
    test_embeddings
)

test_metadata_path = (
    EMBEDDING_ROOT /
    f"{model_slug}_test_metadata.csv"
)

test_metadata.to_csv(
    test_metadata_path,
    index=False
)

print(
    "\nSaved MobileNet test embeddings:"
)

print(
    test_embedding_path
)


# =============================================================================
# 8. GET CLASS IDS
# =============================================================================

prototype_class_ids = (
    prototype_metadata[
        "class_id"
    ]
    .to_numpy()
    .astype(np.int32)
)

test_class_ids = (
    test_metadata[
        "class_id"
    ]
    .to_numpy()
    .astype(np.int32)
)

print(
    "\nPrototype class IDs:",
    prototype_class_ids.shape
)

print(
    "Test class IDs:",
    test_class_ids.shape
)


# =============================================================================
# 9. COMPUTE MOBILENET COSINE SIMILARITY
# =============================================================================
#
# Because both embeddings are L2 normalized:
#
#       cosine_similarity = A @ B.T
#
# Shape:
#
#       2285 test videos × 4764 prototype videos
#
# =============================================================================

print("\n" + "-" * 80)
print("Computing MobileNet cosine similarity")
print("-" * 80)

mobilenet_similarity = (
    test_embeddings
    @ prototype_embeddings.T
).astype(
    np.float32
)

print(
    "Similarity matrix shape:",
    mobilenet_similarity.shape
)

expected_shape = (
    len(test_metadata),
    len(prototype_metadata)
)

if mobilenet_similarity.shape != expected_shape:

    raise RuntimeError(
        f"Unexpected similarity shape. "
        f"Expected {expected_shape}, "
        f"got {mobilenet_similarity.shape}"
    )


# =============================================================================
# 10. SAVE MOBILENET SIMILARITY MATRIX
# =============================================================================

mobilenet_similarity_path = (
    SIMILARITY_ROOT /
    f"{model_slug}_test_vs_prototype_cosine.npy"
)

np.save(
    mobilenet_similarity_path,
    mobilenet_similarity
)

print(
    "\nSaved similarity matrix:"
)

print(
    mobilenet_similarity_path
)


# =============================================================================
# 11. COMPUTE MOBILENET RETRIEVAL METRICS
# =============================================================================

(
    mobilenet_metrics,
    mobilenet_ranks,
    mobilenet_top1_predictions,
    mobilenet_top5_predictions
) = compute_retrieval_metrics(
    mobilenet_similarity,
    prototype_class_ids,
    test_class_ids
)

mobilenet_metrics["model"] = (
    model_name
)

mobilenet_metrics["embedding_dimension"] = (
    expected_dim
)

mobilenet_metrics["num_prototype_videos"] = (
    len(prototype_metadata)
)

mobilenet_metrics["num_test_videos"] = (
    len(test_metadata)
)


# =============================================================================
# 12. CREATE MOBILENET PREDICTION TABLE
# =============================================================================

prediction_rows = []

for i in range(
    len(test_metadata)
):

    ranking = np.argsort(
        -mobilenet_similarity[i]
    )

    ranked_classes = (
        np.array(
            prototype_class_ids
        )[ranking]
    )

    ranked_scores = (
        mobilenet_similarity[
            i,
            ranking
        ]
    )

    true_class = int(
        test_class_ids[i]
    )

    true_rank = int(
        mobilenet_ranks[i]
    )

    prediction_rows.append({

        "uid":
            test_metadata.iloc[i]["uid"],

        "gloss":
            test_metadata.iloc[i]["gloss"],

        "true_class_id":
            true_class,

        "true_rank":
            true_rank,

        "top1_class_id":
            int(
                ranked_classes[0]
            ),

        "top1_gloss":
            class_to_gloss[
                int(
                    ranked_classes[0]
                )
            ],

        "top1_similarity":
            float(
                ranked_scores[0]
            ),

        "top5_class_ids":
            json.dumps(
                [
                    int(x)
                    for x in ranked_classes[:5]
                ]
            ),

        "top5_glosses":
            json.dumps(
                [
                    class_to_gloss[
                        int(x)
                    ]
                    for x in ranked_classes[:5]
                ],
                ensure_ascii=False
            ),

        "top5_similarities":
            json.dumps(
                [
                    float(x)
                    for x in ranked_scores[:5]
                ]
            ),

        "hit_top1":
            bool(
                true_rank <= 1
            ),

        "hit_top5":
            bool(
                true_rank <= 5
            ),

        "hit_top10":
            bool(
                true_rank <= 10
            ),
    })


mobilenet_prediction_df = pd.DataFrame(
    prediction_rows
)

mobilenet_prediction_path = (
    PREDICTION_ROOT /
    f"{model_slug}_official_predictions.csv"
)

mobilenet_prediction_df.to_csv(
    mobilenet_prediction_path,
    index=False
)


# =============================================================================
# 13. SAVE MOBILENET RANKS
# =============================================================================

mobilenet_rank_path = (
    PREDICTION_ROOT /
    f"{model_slug}_ranks.npy"
)

np.save(
    mobilenet_rank_path,
    mobilenet_ranks
)


# =============================================================================
# 14. SAVE MOBILENET METRICS
# =============================================================================

mobilenet_metrics_path = (
    METRIC_ROOT /
    f"{model_slug}_official_metrics.json"
)

with open(
    mobilenet_metrics_path,
    "w"
) as f:

    json.dump(
        mobilenet_metrics,
        f,
        indent=2
    )


# =============================================================================
# 15. DISPLAY MOBILENET RESULTS
# =============================================================================

print("\n" + "=" * 80)
print("MOBILENETV3-LARGE — OFFICIAL CISLR RESULTS")
print("=" * 80)

print(
    f"Top-1       : "
    f"{mobilenet_metrics['Top-1'] * 100:.2f}%"
)

print(
    f"Top-5       : "
    f"{mobilenet_metrics['Top-5'] * 100:.2f}%"
)

print(
    f"Recall@10   : "
    f"{mobilenet_metrics['Recall@10'] * 100:.2f}%"
)

print(
    f"MRR         : "
    f"{mobilenet_metrics['MRR']:.4f}"
)

print(
    f"Mean Rank   : "
    f"{mobilenet_metrics['mean_rank']:.2f}"
)

print(
    f"Median Rank : "
    f"{mobilenet_metrics['median_rank']:.2f}"
)


# =============================================================================
# 16. EFFICIENTNET-B0
# =============================================================================
#
# Now move to EfficientNet.
#
# We DO NOT retrain it.
# We load the already-trained Stage 4.4 checkpoint.
#
# =============================================================================

print("\n\n")
print("#" * 80)
print("# EFFICIENTNET-B0 — OFFICIAL EVALUATION")
print("#" * 80)


model_name = "EfficientNet-B0"

model = models_dict[
    model_name
]

expected_dim = 1280

print(
    "\nEfficientNet expected embedding dimension:",
    expected_dim
)


# =============================================================================
# 17. EXTRACT EFFICIENTNET PROTOTYPE EMBEDDINGS
# =============================================================================

efficientnet_prototype_embeddings, efficientnet_prototype_metadata = (
    extract_split_embeddings(
        model_name,
        model,
        prototype_eval_df,
        "prototype"
    )
)

print(
    "\nEfficientNet prototype embedding shape:",
    efficientnet_prototype_embeddings.shape
)

if (
    efficientnet_prototype_embeddings.shape[1]
    != expected_dim
):

    raise RuntimeError(
        f"EfficientNet-B0 embedding dimension mismatch. "
        f"Expected {expected_dim}, got "
        f"{efficientnet_prototype_embeddings.shape[1]}"
    )


# =============================================================================
# 18. NORMALIZE EFFICIENTNET PROTOTYPES
# =============================================================================

efficientnet_prototype_norms = np.linalg.norm(
    efficientnet_prototype_embeddings,
    axis=1
)

if not np.allclose(
    efficientnet_prototype_norms,
    1.0,
    atol=1e-4
):

    efficientnet_prototype_embeddings = (
        efficientnet_prototype_embeddings /
        np.maximum(
            efficientnet_prototype_norms[:, None],
            1e-12
        )
    )


# =============================================================================
# 19. SAVE EFFICIENTNET PROTOTYPES
# =============================================================================

efficientnet_model_slug = (
    model_name
    .lower()
    .replace("-", "_")
    .replace(" ", "_")
)

efficientnet_prototype_embedding_path = (
    EMBEDDING_ROOT /
    f"{efficientnet_model_slug}_prototype_embeddings.npy"
)

np.save(
    efficientnet_prototype_embedding_path,
    efficientnet_prototype_embeddings
)

efficientnet_prototype_metadata_path = (
    EMBEDDING_ROOT /
    f"{efficientnet_model_slug}_prototype_metadata.csv"
)

efficientnet_prototype_metadata.to_csv(
    efficientnet_prototype_metadata_path,
    index=False
)

print(
    "\nSaved EfficientNet prototype embeddings:"
)

print(
    efficientnet_prototype_embedding_path
)


# =============================================================================
# 20. EXTRACT EFFICIENTNET TEST EMBEDDINGS
# =============================================================================

print("\n" + "-" * 80)
print("Extracting EfficientNet-B0 TEST embeddings")
print("-" * 80)

efficientnet_test_embeddings, efficientnet_test_metadata = (
    extract_split_embeddings(
        model_name,
        model,
        test_eval_df,
        "test"
    )
)

print(
    "\nEfficientNet test embedding shape:",
    efficientnet_test_embeddings.shape
)

if (
    efficientnet_test_embeddings.shape[1]
    != expected_dim
):

    raise RuntimeError(
        f"EfficientNet-B0 test embedding dimension mismatch. "
        f"Expected {expected_dim}, got "
        f"{efficientnet_test_embeddings.shape[1]}"
    )


# =============================================================================
# 21. NORMALIZE EFFICIENTNET TEST EMBEDDINGS
# =============================================================================

efficientnet_test_norms = np.linalg.norm(
    efficientnet_test_embeddings,
    axis=1
)

if not np.allclose(
    efficientnet_test_norms,
    1.0,
    atol=1e-4
):

    efficientnet_test_embeddings = (
        efficientnet_test_embeddings /
        np.maximum(
            efficientnet_test_norms[:, None],
            1e-12
        )
    )


# =============================================================================
# 22. SAVE EFFICIENTNET TEST EMBEDDINGS
# =============================================================================

efficientnet_test_embedding_path = (
    EMBEDDING_ROOT /
    f"{efficientnet_model_slug}_test_embeddings.npy"
)

np.save(
    efficientnet_test_embedding_path,
    efficientnet_test_embeddings
)

efficientnet_test_metadata_path = (
    EMBEDDING_ROOT /
    f"{efficientnet_model_slug}_test_metadata.csv"
)

efficientnet_test_metadata.to_csv(
    efficientnet_test_metadata_path,
    index=False
)


# =============================================================================
# 23. EFFICIENTNET CLASS IDS
# =============================================================================

efficientnet_prototype_class_ids = (
    efficientnet_prototype_metadata[
        "class_id"
    ]
    .to_numpy()
    .astype(np.int32)
)

efficientnet_test_class_ids = (
    efficientnet_test_metadata[
        "class_id"
    ]
    .to_numpy()
    .astype(np.int32)
)


# =============================================================================
# 24. EFFICIENTNET COSINE SIMILARITY
# =============================================================================

print("\n" + "-" * 80)
print("Computing EfficientNet cosine similarity")
print("-" * 80)

efficientnet_similarity = (
    efficientnet_test_embeddings
    @ efficientnet_prototype_embeddings.T
).astype(
    np.float32
)

print(
    "Similarity matrix shape:",
    efficientnet_similarity.shape
)

expected_shape = (
    len(efficientnet_test_metadata),
    len(efficientnet_prototype_metadata)
)

if efficientnet_similarity.shape != expected_shape:

    raise RuntimeError(
        f"Unexpected EfficientNet similarity shape. "
        f"Expected {expected_shape}, "
        f"got {efficientnet_similarity.shape}"
    )


# =============================================================================
# 25. SAVE EFFICIENTNET SIMILARITY
# =============================================================================

efficientnet_similarity_path = (
    SIMILARITY_ROOT /
    f"{efficientnet_model_slug}_test_vs_prototype_cosine.npy"
)

np.save(
    efficientnet_similarity_path,
    efficientnet_similarity
)


# =============================================================================
# 26. EFFICIENTNET METRICS
# =============================================================================

(
    efficientnet_metrics,
    efficientnet_ranks,
    efficientnet_top1_predictions,
    efficientnet_top5_predictions
) = compute_retrieval_metrics(
    efficientnet_similarity,
    efficientnet_prototype_class_ids,
    efficientnet_test_class_ids
)

efficientnet_metrics["model"] = (
    model_name
)

efficientnet_metrics["embedding_dimension"] = (
    expected_dim
)

efficientnet_metrics["num_prototype_videos"] = (
    len(efficientnet_prototype_metadata)
)

efficientnet_metrics["num_test_videos"] = (
    len(efficientnet_test_metadata)
)


# =============================================================================
# 27. EFFICIENTNET PREDICTION TABLE
# =============================================================================

efficientnet_prediction_rows = []

for i in range(
    len(efficientnet_test_metadata)
):

    ranking = np.argsort(
        -efficientnet_similarity[i]
    )

    ranked_classes = (
        np.array(
            efficientnet_prototype_class_ids
        )[ranking]
    )

    ranked_scores = (
        efficientnet_similarity[
            i,
            ranking
        ]
    )

    true_class = int(
        efficientnet_test_class_ids[i]
    )

    true_rank = int(
        efficientnet_ranks[i]
    )

    efficientnet_prediction_rows.append({

        "uid":
            efficientnet_test_metadata.iloc[i]["uid"],

        "gloss":
            efficientnet_test_metadata.iloc[i]["gloss"],

        "true_class_id":
            true_class,

        "true_rank":
            true_rank,

        "top1_class_id":
            int(
                ranked_classes[0]
            ),

        "top1_gloss":
            class_to_gloss[
                int(
                    ranked_classes[0]
                )
            ],

        "top1_similarity":
            float(
                ranked_scores[0]
            ),

        "top5_class_ids":
            json.dumps(
                [
                    int(x)
                    for x in ranked_classes[:5]
                ]
            ),

        "top5_glosses":
            json.dumps(
                [
                    class_to_gloss[
                        int(x)
                    ]
                    for x in ranked_classes[:5]
                ],
                ensure_ascii=False
            ),

        "top5_similarities":
            json.dumps(
                [
                    float(x)
                    for x in ranked_scores[:5]
                ]
            ),

        "hit_top1":
            bool(
                true_rank <= 1
            ),

        "hit_top5":
            bool(
                true_rank <= 5
            ),

        "hit_top10":
            bool(
                true_rank <= 10
            ),
    })


efficientnet_prediction_df = pd.DataFrame(
    efficientnet_prediction_rows
)

efficientnet_prediction_path = (
    PREDICTION_ROOT /
    f"{efficientnet_model_slug}_official_predictions.csv"
)

efficientnet_prediction_df.to_csv(
    efficientnet_prediction_path,
    index=False
)


# =============================================================================
# 28. SAVE EFFICIENTNET RANKS
# =============================================================================

efficientnet_rank_path = (
    PREDICTION_ROOT /
    f"{efficientnet_model_slug}_ranks.npy"
)

np.save(
    efficientnet_rank_path,
    efficientnet_ranks
)


# =============================================================================
# 29. SAVE EFFICIENTNET METRICS
# =============================================================================

efficientnet_metrics_path = (
    METRIC_ROOT /
    f"{efficientnet_model_slug}_official_metrics.json"
)

with open(
    efficientnet_metrics_path,
    "w"
) as f:

    json.dump(
        efficientnet_metrics,
        f,
        indent=2
    )


# =============================================================================
# 30. DISPLAY EFFICIENTNET RESULTS
# =============================================================================

print("\n" + "=" * 80)
print("EFFICIENTNET-B0 — OFFICIAL CISLR RESULTS")
print("=" * 80)

print(
    f"Top-1       : "
    f"{efficientnet_metrics['Top-1'] * 100:.2f}%"
)

print(
    f"Top-5       : "
    f"{efficientnet_metrics['Top-5'] * 100:.2f}%"
)

print(
    f"Recall@10   : "
    f"{efficientnet_metrics['Recall@10'] * 100:.2f}%"
)

print(
    f"MRR         : "
    f"{efficientnet_metrics['MRR']:.4f}"
)

print(
    f"Mean Rank   : "
    f"{efficientnet_metrics['mean_rank']:.2f}"
)

print(
    f"Median Rank : "
    f"{efficientnet_metrics['median_rank']:.2f}"
)


# =============================================================================
# 31. BUILD FINAL MODEL COMPARISON
# =============================================================================
#
# We preserve the ResNet18 metrics if the previous cell successfully
# completed its evaluation before reaching MobileNet.
#
# =============================================================================

final_results = {}

# -------------------------------------------------------------------------
# Recover ResNet18 metrics from saved JSON if available
# -------------------------------------------------------------------------

resnet_metrics_path = (
    METRIC_ROOT /
    "resnet18_official_metrics.json"
)

if resnet_metrics_path.exists():

    with open(
        resnet_metrics_path,
        "r"
    ) as f:

        resnet_metrics = json.load(f)

    final_results["ResNet18"] = (
        resnet_metrics
    )

    print(
        "\n[OK] ResNet18 metrics recovered from disk."
    )

else:

    # If the previous cell had not saved ResNet metrics,
    # do NOT rerun ResNet extraction here.
    print(
        "\n[WARNING] ResNet18 metrics file was not found."
    )

    print(
        "ResNet18 evaluation is not rerun in this continuation cell."
    )


# -------------------------------------------------------------------------
# MobileNet
# -------------------------------------------------------------------------

final_results[
    "MobileNetV3-Large"
] = mobilenet_metrics


# -------------------------------------------------------------------------
# EfficientNet
# -------------------------------------------------------------------------

final_results[
    "EfficientNet-B0"
] = efficientnet_metrics


# =============================================================================
# 32. FINAL COMPARISON DATAFRAME
# =============================================================================

comparison_rows = []

for name, metrics in final_results.items():

    comparison_rows.append({

        "model":
            name,

        "embedding_dimension":
            metrics[
                "embedding_dimension"
            ],

        "num_prototype_videos":
            metrics[
                "num_prototype_videos"
            ],

        "num_test_videos":
            metrics[
                "num_test_videos"
            ],

        "Top-1":
            metrics[
                "Top-1"
            ],

        "Top-5":
            metrics[
                "Top-5"
            ],

        "Recall@10":
            metrics[
                "Recall@10"
            ],

        "MRR":
            metrics[
                "MRR"
            ],

        "mean_rank":
            metrics[
                "mean_rank"
            ],

        "median_rank":
            metrics[
                "median_rank"
            ],
    })


comparison_df = pd.DataFrame(
    comparison_rows
)

comparison_df = comparison_df.sort_values(
    by="Top-1",
    ascending=False
).reset_index(
    drop=True
)


# =============================================================================
# 33. SAVE FINAL COMPARISON
# =============================================================================

comparison_csv_path = (
    STAGE45_ROOT /
    "stage4_5_official_model_comparison.csv"
)

comparison_df.to_csv(
    comparison_csv_path,
    index=False
)


comparison_json_path = (
    STAGE45_ROOT /
    "stage4_5_official_model_comparison.json"
)

with open(
    comparison_json_path,
    "w"
) as f:

    json.dump(
        final_results,
        f,
        indent=2
    )


# =============================================================================
# 34. FINAL RESULTS
# =============================================================================

print("\n\n")
print("#" * 80)
print("# STAGE 4.5 — OFFICIAL CISLR EVALUATION")
print("# FINAL COMPARISON")
print("#" * 80)

display_df = comparison_df.copy()

display_df["Top-1"] = (
    display_df["Top-1"] * 100
).round(2)

display_df["Top-5"] = (
    display_df["Top-5"] * 100
).round(2)

display_df["Recall@10"] = (
    display_df["Recall@10"] * 100
).round(2)

display_df["MRR"] = (
    display_df["MRR"]
).round(4)

display_df["mean_rank"] = (
    display_df["mean_rank"]
).round(2)

display_df["median_rank"] = (
    display_df["median_rank"]
).round(2)

print(
    display_df.to_string(
        index=False
    )
)


# =============================================================================
# 35. FINAL ARTIFACTS
# =============================================================================

print("\n" + "=" * 80)
print("STAGE 4.5 CONTINUATION COMPLETE")
print("=" * 80)

print("\nMobileNet artifacts:")
print(
    "  Prototype embeddings :",
    prototype_embedding_path
)
print(
    "  Test embeddings      :",
    test_embedding_path
)
print(
    "  Similarity matrix    :",
    mobilenet_similarity_path
)
print(
    "  Predictions          :",
    mobilenet_prediction_path
)
print(
    "  Metrics              :",
    mobilenet_metrics_path
)


print("\nEfficientNet artifacts:")
print(
    "  Prototype embeddings :",
    efficientnet_prototype_embedding_path
)
print(
    "  Test embeddings      :",
    efficientnet_test_embedding_path
)
print(
    "  Similarity matrix    :",
    efficientnet_similarity_path
)
print(
    "  Predictions          :",
    efficientnet_prediction_path
)
print(
    "  Metrics              :",
    efficientnet_metrics_path
)


print("\nFinal comparison:")
print(
    "  CSV  :",
    comparison_csv_path
)
print(
    "  JSON :",
    comparison_json_path
)

print("\n" + "=" * 80)
print("DONE")
print("=" * 80)

STAGE 4.5 — CONTINUATION

Correct embedding dimensions:
  ResNet18          : 512
  MobileNetV3-Large : 960
  EfficientNet-B0   : 1280

[OK] Previous Stage 4.5 state is available.

MOBILENETV3-LARGE — CONTINUING

Existing MobileNet prototype embedding shape:
(4764, 960)
[OK] MobileNetV3-Large embedding dimension = 960
[OK] MobileNet prototype embeddings are L2 normalized.

Saved MobileNet prototype embeddings:
/home/dipshikha/Music/cao/CISLR_RESEARCH/audit/stage4_5_official_evaluation/embeddings/mobilenetv3_large_prototype_embeddings.npy

--------------------------------------------------------------------------------
Extracting MobileNetV3-Large TEST embeddings
--------------------------------------------------------------------------------

EXTRACTING MobileNetV3-Large — TEST


MobileNetV3-Large / test:   0%|          | 0/2285 [00:00<?, ?it/s]

[ROBUST DECODE] 08E4Dr0zyKk_2.mp4 | reported=78 | decoded=69 | using actual decoded frames
[ROBUST DECODE] 4kKAKsQ5mGU_2.mp4 | reported=146 | decoded=139 | using actual decoded frames
[ROBUST DECODE] 8NqmODtSLgI_2.mp4 | reported=90 | decoded=84 | using actual decoded frames
[ROBUST DECODE] BSezNX5k4dA_2.mp4 | reported=76 | decoded=67 | using actual decoded frames
[ROBUST DECODE] CRdSfDtdSj8_2.mp4 | reported=148 | decoded=145 | using actual decoded frames
[ROBUST DECODE] CcX4eD-UArI_2.mp4 | reported=166 | decoded=160 | using actual decoded frames
[ROBUST DECODE] CjZKfMv3QsA_2.mp4 | reported=109 | decoded=102 | using actual decoded frames
[ROBUST DECODE] Dg1FJhcTg2E_2.mp4 | reported=87 | decoded=85 | using actual decoded frames
[ROBUST DECODE] DlyWuMwYeVc_2.mp4 | reported=147 | decoded=145 | using actual decoded frames
[ROBUST DECODE] EI4ConsbYws_2.mp4 | reported=123 | decoded=115 | using actual decoded frames
[ROBUST DECODE] EmUsRBX9664_2.mp4 | reported=95 | decoded=89 | using actual de

EfficientNet-B0 / prototype:   0%|          | 0/4764 [00:00<?, ?it/s]

[ROBUST DECODE] -0XujLX2_9I_2.mp4 | reported=70 | decoded=58 | using actual decoded frames
[ROBUST DECODE] -2_NUxX4PVA_2.mp4 | reported=119 | decoded=107 | using actual decoded frames
[ROBUST DECODE] -5p4hn8J87A_2.mp4 | reported=78 | decoded=75 | using actual decoded frames
[ROBUST DECODE] -I0sjwzg0tY_2.mp4 | reported=197 | decoded=188 | using actual decoded frames
[ROBUST DECODE] 039-5dIML0I_2.mp4 | reported=103 | decoded=99 | using actual decoded frames
[ROBUST DECODE] 0ETSwMTLaac_2.mp4 | reported=98 | decoded=97 | using actual decoded frames
[ROBUST DECODE] 0F_szwrQs1g_2.mp4 | reported=87 | decoded=79 | using actual decoded frames
[ROBUST DECODE] 0JIsxA88oss_2.mp4 | reported=106 | decoded=100 | using actual decoded frames
[ROBUST DECODE] 0Qk8-_ZEmHM_2.mp4 | reported=96 | decoded=90 | using actual decoded frames
[ROBUST DECODE] 0fdRLmUIlZE_2.mp4 | reported=131 | decoded=125 | using actual decoded frames
[ROBUST DECODE] 0ijcVXYEvY8_2.mp4 | reported=112 | decoded=109 | using actual dec

EfficientNet-B0 / test:   0%|          | 0/2285 [00:00<?, ?it/s]

[ROBUST DECODE] 08E4Dr0zyKk_2.mp4 | reported=78 | decoded=69 | using actual decoded frames
[ROBUST DECODE] 4kKAKsQ5mGU_2.mp4 | reported=146 | decoded=139 | using actual decoded frames
[ROBUST DECODE] 8NqmODtSLgI_2.mp4 | reported=90 | decoded=84 | using actual decoded frames
[ROBUST DECODE] BSezNX5k4dA_2.mp4 | reported=76 | decoded=67 | using actual decoded frames
[ROBUST DECODE] CRdSfDtdSj8_2.mp4 | reported=148 | decoded=145 | using actual decoded frames
[ROBUST DECODE] CcX4eD-UArI_2.mp4 | reported=166 | decoded=160 | using actual decoded frames
[ROBUST DECODE] CjZKfMv3QsA_2.mp4 | reported=109 | decoded=102 | using actual decoded frames
[ROBUST DECODE] Dg1FJhcTg2E_2.mp4 | reported=87 | decoded=85 | using actual decoded frames
[ROBUST DECODE] DlyWuMwYeVc_2.mp4 | reported=147 | decoded=145 | using actual decoded frames
[ROBUST DECODE] EI4ConsbYws_2.mp4 | reported=123 | decoded=115 | using actual decoded frames
[ROBUST DECODE] EmUsRBX9664_2.mp4 | reported=95 | decoded=89 | using actual de